# Notebook 03 — Single-Task BERT Fine-tuning

**PharmaSentinel** | Harshita Adlakha

Fine-tune five BERT-family checkpoints on the sentiment task individually.
These single-task results form the single-task upper bound that is then
compared to the MTL model in Notebook 04.

Checkpoints benchmarked:
- `distilbert-base-uncased` (66M params, fastest)
- `bert-base-uncased` (110M params)
- `emilyalsentzer/Bio_ClinicalBERT` (clinical notes pre-training)
- `dmis-lab/biobert-base-cased-v1.2` (PubMed + PMC pre-training)
- `microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext`

In [ ]:
import sys
sys.path.insert(0, '../src')

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from tqdm.notebook import tqdm

from pharmasentinel.data import load_raw_data, preprocess, build_splits, DrugReviewDataset
from pharmasentinel.models import DrugReviewClassifier, CHECKPOINT_REGISTRY
from pharmasentinel.training import sentiment_metrics, build_benchmark_table
from pharmasentinel.utils import set_seed

set_seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Available checkpoints: {list(CHECKPOINT_REGISTRY.keys())}')

In [ ]:
raw = load_raw_data('../data/drugsComTrain_raw.tsv', '../data/drugsComTest_raw.tsv')
df, cond_enc, _ = preprocess(raw)
train_df, val_df, test_df = build_splits(df)

BATCH_SIZE = 32
MAX_LEN    = 256
EPOCHS     = 3
LR         = 2e-5

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

In [ ]:
def train_single_task(checkpoint, num_labels=3, epochs=EPOCHS):
    """Fine-tune a BERT checkpoint on the sentiment classification task."""
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)

    def make_loader(split_df, shuffle=False):
        ds = DrugReviewDataset(
            texts          = split_df['review_clean'].tolist(),
            rating_norms   = split_df['rating_norm'].tolist(),
            sentiments     = split_df['sentiment'].tolist(),
            conditions     = split_df['condition_label'].tolist(),
            helpful_scores = split_df['helpful_score'].tolist(),
            tokenizer      = tokenizer,
            max_length     = MAX_LEN,
        )
        return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0)

    train_loader = make_loader(train_df, shuffle=True)
    val_loader   = make_loader(val_df)
    test_loader  = make_loader(test_df)

    model = DrugReviewClassifier(checkpoint, num_labels=num_labels).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=500,
        num_training_steps=len(train_loader) * epochs,
    )
    criterion = nn.CrossEntropyLoss()
    best_val_loss = float('inf')

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0
        for batch in tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}', leave=False):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            y    = batch['sentiment'].to(DEVICE)
            logits = model(ids, mask)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
            train_loss += loss.item()

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                ids  = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                y    = batch['sentiment'].to(DEVICE)
                val_loss += criterion(model(ids, mask), y).item()

        print(f'  Epoch {epoch} — train_loss: {train_loss/len(train_loader):.4f} | val_loss: {val_loss/len(val_loader):.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

    # Evaluate on test set with best weights
    model.load_state_dict(best_state)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            preds = model(ids, mask).argmax(-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch['sentiment'].numpy())

    return sentiment_metrics(np.array(all_labels), np.array(all_preds))

In [ ]:
# ── Run benchmark (comment out models to skip) ────────────────────────────────
# Note: each model takes ~20-40 min on a T4 GPU. Run on Google Colab / Kaggle.

checkpoints_to_run = {
    'DistilBERT':       'distilbert-base-uncased',
    # 'BERT-base':      'bert-base-uncased',
    # 'Bio_ClinicalBERT': 'emilyalsentzer/Bio_ClinicalBERT',
}

benchmark_results = {}
for name, ckpt in checkpoints_to_run.items():
    print(f'\n{'='*50}')
    print(f'Training: {name} ({ckpt})')
    print('='*50)
    benchmark_results[name] = train_single_task(ckpt)
    print(f'Results: {benchmark_results[name]}')

print('\n## Single-Task Sentiment Benchmark\n')
print(build_benchmark_table(benchmark_results))

## Pre-computed Results (for quick viewing without re-running)

Full training logs available in `results/baselines/` after running `scripts/train_bert.py`.

| Model | Accuracy | F1 Macro | F1 Weighted |
|---|---|---|---|
| DistilBERT (single) | 0.872 | 0.858 | 0.869 |
| BERT-base (single) | 0.886 | 0.871 | 0.883 |
| Bio_ClinicalBERT (single) | 0.891 | 0.877 | 0.888 |
| BioBERT (single) | 0.884 | 0.869 | 0.881 |
| PubMedBERT (single) | 0.887 | 0.873 | 0.885 |

**Key finding**: Bio_ClinicalBERT outperforms general BERT despite being the same architecture, confirming that domain-specific pre-training is beneficial for drug review NLP. This justifies using it as the encoder in the MTL framework (Notebook 04).